![title](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/journey.png)

![sponsors](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/sponsors.jpg)

![r](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/profiles.png)

![spark_vs_hadoop](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/spark_vs_hadoop.png)

![clusters](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/clusters.png)

# AQE off

In [1]:
# Turn AQE fully off so stage/task counts match the theory
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")   # make the default explicit

# verify
print("aqe:     ", spark.conf.get("spark.sql.adaptive.enabled"))
print("coalesce:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 3, Finished, Available, Finished, False)

aqe:      false
coalesce: false


# RDD (the "HOW") = you spell out the steps

## No schema, no columns, just elements

In [2]:
lines  = sc.parallelize(["hello world", "hello spark"])
words  = lines.flatMap(lambda line: line.split(" "))   
pairs  = words.map(lambda w: (w, 1))                   
counts = pairs.reduceByKey(lambda a, b: a + b)         
counts.collect()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 5, Finished, Available, Finished, False)

[('spark', 1), ('world', 1), ('hello', 2)]

# DataFrame (the "WHAT") = you declare what you want and Spark figures out the steps
## Less code, a schema Spark understands

In [3]:
df = spark.createDataFrame(
    [("hello",), ("world",), ("hello",), ("spark",)],
    ["word"]
)
counts = df.groupBy("word").count()   # bind counts to THIS df
counts.collect()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 6, Finished, Available, Finished, False)

[Row(word='hello', count=2),
 Row(word='spark', count=1),
 Row(word='world', count=1)]

## How many partitions?

In [4]:
print("partitions:", df.rdd.getNumPartitions())

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 7, Finished, Available, Finished, False)

partitions: 8


## How many rows land in each partition?

In [5]:
from pyspark.sql import functions as F

rows_per_partition = (df
    .withColumn("pid", F.spark_partition_id())
    .groupBy("pid")
    .count()
    .orderBy("pid"))
rows_per_partition.show()


StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 8, Finished, Available, Finished, False)

+---+-----+
|pid|count|
+---+-----+
|  1|    1|
|  3|    1|
|  5|    1|
|  7|    1|
+---+-----+



## Seeing the empty partitions

In [6]:
print(df.rdd.glom().map(len).collect())

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 10, Finished, Available, Finished, False)

[0, 1, 0, 1, 0, 1, 0, 1]


![spark_vs_hadoop](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/jobs_tasks.png)

### 

![spark_vs_hadoop](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/narrow_wide_transformations.png)

![actions](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/actions.png)

![shuffle_boundary](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/shuffle_boundary.png)

# AQE off

In [ ]:
# Turn AQE fully off so stage/task counts match the theory
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")   # make the default explicit

# verify
print("aqe:     ", spark.conf.get("spark.sql.adaptive.enabled"))
print("coalesce:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))


# The simplest possible case: 1 job, 1 stage, AQE off

## Read CSV File

In [7]:
# Read user.csv file, nr of rows= 20000
csv_path = "Files/orders.csv"

orders_df = (spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(csv_path))

display(orders_df)

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40ffe257-ca22-4b82-a697-546e3f022c33)

## Transforming the data

In [8]:
from pyspark.sql import functions as F

cancelled_orders_df = (orders_df
                .filter(F.col("status") == "CANCELLED")        # narrow transformation -> lazy - nothings runs
                .select("user_id", "amount", "status"))        # narrow transformation -> lazy - nothings runs
cancelled_orders_df.collect()    # ACTION -> now the job runs -> 1 job, 1 stage
# In the Spark UI: one job, one stage, N tasks (= partitions). filter + select are fused inside each task (WholeStageCodegen) -> no shuffle, no second stage.

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 12, Finished, Available, Finished, False)

[Row(user_id=384, amount=194.26, status='CANCELLED'),
 Row(user_id=486, amount=365.87, status='CANCELLED'),
 Row(user_id=136, amount=268.38, status='CANCELLED'),
 Row(user_id=260, amount=344.97, status='CANCELLED'),
 Row(user_id=337, amount=212.35, status='CANCELLED'),
 Row(user_id=394, amount=261.97, status='CANCELLED'),
 Row(user_id=317, amount=357.69, status='CANCELLED'),
 Row(user_id=10, amount=362.92, status='CANCELLED'),
 Row(user_id=67, amount=33.92, status='CANCELLED'),
 Row(user_id=211, amount=143.27, status='CANCELLED'),
 Row(user_id=451, amount=237.15, status='CANCELLED'),
 Row(user_id=20, amount=356.84, status='CANCELLED'),
 Row(user_id=291, amount=298.34, status='CANCELLED'),
 Row(user_id=7, amount=241.0, status='CANCELLED'),
 Row(user_id=434, amount=154.29, status='CANCELLED'),
 Row(user_id=430, amount=367.55, status='CANCELLED'),
 Row(user_id=405, amount=77.72, status='CANCELLED'),
 Row(user_id=121, amount=375.85, status='CANCELLED'),
 Row(user_id=144, amount=365.78, sta

# Add a wide transformation: 1 Job, 2 Stages

## The Transformations

In [9]:
from pyspark.sql import functions as F

cancelled_orders_df = (orders_df
                .filter(F.col("status") == "CANCELLED")        # narrow transformation 
                .select("user_id", "amount", "status")        # narrow transformation > all fused into stage 1
                .groupBy("user_id").agg(F.sum("amount")))      # WIDE -> shuffle -> stage 2

# No action yet -> check the Spark UI: no job appeared.   

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 13, Finished, Available, Finished, False)

## The Action

In [10]:
cancelled_orders_df.collect()   # ACTION -> 1 job, 2 stages
# In the Spark UI: one job, two stage, N tasks (= partitions)

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 14, Finished, Available, Finished, False)

[Row(user_id=471, sum(amount)=2642.56),
 Row(user_id=148, sum(amount)=2175.1600000000003),
 Row(user_id=463, sum(amount)=1839.43),
 Row(user_id=496, sum(amount)=2789.87),
 Row(user_id=392, sum(amount)=2960.02),
 Row(user_id=243, sum(amount)=2336.8299999999995),
 Row(user_id=31, sum(amount)=2498.2200000000003),
 Row(user_id=451, sum(amount)=2840.92),
 Row(user_id=251, sum(amount)=2831.96),
 Row(user_id=85, sum(amount)=2839.3900000000003),
 Row(user_id=137, sum(amount)=1849.6499999999999),
 Row(user_id=458, sum(amount)=1666.4700000000003),
 Row(user_id=65, sum(amount)=2516.52),
 Row(user_id=53, sum(amount)=2249.13),
 Row(user_id=255, sum(amount)=3081.9500000000003),
 Row(user_id=481, sum(amount)=1880.1299999999999),
 Row(user_id=296, sum(amount)=2958.6),
 Row(user_id=472, sum(amount)=2076.1600000000003),
 Row(user_id=133, sum(amount)=2279.2400000000002),
 Row(user_id=78, sum(amount)=1402.22),
 Row(user_id=322, sum(amount)=1649.39),
 Row(user_id=362, sum(amount)=2356.69),
 Row(user_id=321

In [11]:
# actual row distribution across the post-shuffle partitions
from pyspark.sql.functions import spark_partition_id

(cancelled_orders_df.withColumn("pid", spark_partition_id())
       .groupBy("pid").count()
       .orderBy("count", ascending=False)
       .show(10))

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 15, Finished, Available, Finished, False)

+---+-----+
|pid|count|
+---+-----+
|101|    6|
| 32|    6|
|113|    6|
|112|    6|
|103|    6|
|153|    5|
| 30|    5|
|184|    5|
|118|    5|
|198|    5|
+---+-----+
only showing top 10 rows


![file_types](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/file_types.png)

![azy_eval](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/lazy_eval_optimizations.png)

# Optimization example: Partition Pruning

## Read a plain CSV and Convert it to Delta (Parquet under the hood)

In [12]:
# Read user.csv file, nr of rows= 20 000
csv_path = "Files/orders.csv"

orders_df = (spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(csv_path))

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 16, Finished, Available, Finished, False)

## Partition the table by status

In [13]:
# Write orders_df as delta table
(orders_df.write.mode("overwrite")
    .partitionBy("status")
    .format("delta")
    .saveAsTable("orders_part"))

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 17, Finished, Available, Finished, False)

## Partition pruning: skip whole folders

In [14]:
# query one status -> Spark reads only that folder

from pyspark.sql import functions as F

part_orders = spark.read.table("orders_part")
cancelled_part_orders_df = (part_orders
    .filter(F.col("status") == "CANCELLED")
    .select("user_id", "amount"))

# cancelled_part_orders_df.explain(mode="formatted")   # show the plan (no job)
cancelled_part_orders_df.collect() 

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 18, Finished, Available, Finished, False)

[Row(user_id=384, amount=194.26),
 Row(user_id=486, amount=365.87),
 Row(user_id=136, amount=268.38),
 Row(user_id=260, amount=344.97),
 Row(user_id=337, amount=212.35),
 Row(user_id=394, amount=261.97),
 Row(user_id=317, amount=357.69),
 Row(user_id=10, amount=362.92),
 Row(user_id=67, amount=33.92),
 Row(user_id=211, amount=143.27),
 Row(user_id=451, amount=237.15),
 Row(user_id=20, amount=356.84),
 Row(user_id=291, amount=298.34),
 Row(user_id=7, amount=241.0),
 Row(user_id=434, amount=154.29),
 Row(user_id=430, amount=367.55),
 Row(user_id=405, amount=77.72),
 Row(user_id=121, amount=375.85),
 Row(user_id=144, amount=365.78),
 Row(user_id=338, amount=25.04),
 Row(user_id=15, amount=356.32),
 Row(user_id=494, amount=69.71),
 Row(user_id=143, amount=141.63),
 Row(user_id=310, amount=146.06),
 Row(user_id=473, amount=227.93),
 Row(user_id=163, amount=224.33),
 Row(user_id=18, amount=289.86),
 Row(user_id=171, amount=356.31),
 Row(user_id=8, amount=369.07),
 Row(user_id=29, amount=313.

## Prep for later

In [15]:
from pyspark.sql import functions as F

# small dimension table: users (500 rows)
users = (spark.read.option("header", True).option("inferSchema", True)
         .csv("Files/users.csv"))
users.write.mode("overwrite").format("delta").saveAsTable("users")

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 19, Finished, Available, Finished, False)

In [16]:
# larger fact table: orders — blow it up to ~2 million rows so it's clearly "big"
orders_big = (spark.read.option("header", True).option("inferSchema", True)
              .csv("Files/orders.csv"))
# repeat the 20k rows ~100x to simulate a large fact table
orders_big = (orders_big
              .crossJoin(spark.range(100000).select(F.col("id").alias("dup")))
              .drop("dup"))        # remove the counter column, keep only real order columns
orders_big.write.mode("overwrite").format("delta").saveAsTable("orders_big")

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 20, Finished, Available, Finished, False)

![bhj](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/broadcast_hash_join.png)

![shj](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/shuffle_hash_join.png)

![sort_merge_join](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/sort_merge_join.png)

![nlj](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/nested_loop_join.png)

# How to check a table's size to decide on a join strategy

## Load table users from csv

## Load large fact table orders from csv

## Read the size back and decide (DESCRIBE DETAIL)

In [17]:
from pyspark.sql import functions as F

for t in ["users", "orders_big"]:
    d = spark.sql(f"DESCRIBE DETAIL {t}").select("sizeInBytes", "numFiles").first()
    size_mb = d["sizeInBytes"] / 1024 / 1024
    print(f"{t}: {size_mb:.2f} MB across {d['numFiles']} files")

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 21, Finished, Available, Finished, False)

users: 0.01 MB across 1 files
orders_big: 9590.42 MB across 8 files


## Broadcast the users table

In [18]:
from pyspark.sql.functions import broadcast

orders_big = spark.read.table("orders_big")
users = spark.read.table("users")

# small side (users) broadcast to every executor; big side (orders_big) stays put
bhj = orders_big.join(users.hint("BROADCAST"), "user_id")

#bhj.explain(mode="formatted")   # confirm the strategy in the plan
bhj.write.format("noop").mode("overwrite").save()                   # ACTION -> runs the join

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 22, Finished, Available, Finished, False)

## Force a sort-merge join (same tables, different strategy)

In [ ]:
orders = spark.read.table("orders_big")
users  = spark.read.table("users")

# hint MERGE to force sort-merge instead of broadcast
smj = orders.join(users.hint("MERGE"), "user_id")

#smj.explain(mode="formatted")
smj.write.format("noop").mode("overwrite").save()

![aqe](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/performance_guidelines.png)

<div style="font-size: 1.4em;">

<h2>Broadcast vs Sort-Merge — same join, same result, different execution</h2>

<table>
<tr><th></th><th>Broadcast Hash Join</th><th>Sort-Merge Join</th></tr>
<tr><td><b>Join operator</b></td><td><code>BroadcastHashJoin</code></td><td><code>SortMergeJoin</code></td></tr>
<tr><td><b>Shuffle</b></td><td>none — big table stays put</td><td>both sides shuffled (two <code>Exchange</code>)</td></tr>
<tr><td><b>Sort</b></td><td>none</td><td>both sides sorted (two <code>Sort</code>)</td></tr>
<tr><td><b>Small table</b></td><td>copied to every executor (<code>BroadcastExchange</code>)</td><td>shuffled like any other</td></tr>
<tr><td><b>Stages</b></td><td>~one (no shuffle boundary)</td><td>more — each shuffle adds a stage</td></tr>
<tr><td><b>Best when</b></td><td>one side &lt; ~10 MB</td><td>both sides large, neither broadcastable</td></tr>
</table>


<p></p>
<p style="font-size: 1.5em;">Here <code>users</code> is ~0.1 MB, so broadcast is right: it skips all the shuffling and sorting. Sort-merge does that same work for no benefit — the plan shows exactly what it added.</p>
</div>


# A filter leaves partitions nearly empty

In [19]:
from pyspark.sql import functions as F

orders = (spark.read.option("header", True).option("inferSchema", True)
              .csv("Files/orders.csv"))

orders = orders.repartition(16)  
# filter to something RARE -> now partitions really are near-empty
rare_orders = orders.filter(F.col("status") == "PENDING")  
print("partitions:", rare_orders.rdd.getNumPartitions())    # still 16

(rare_orders.withColumn("pid", F.spark_partition_id())
     .groupBy("pid").count().orderBy("pid").show())
# now you'll see counts like 400, 412, 414... -> THAT is "too tiny"

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 23, Finished, Available, Finished, False)

partitions: 16
+---+-----+
|pid|count|
+---+-----+
|  0|  414|
|  1|  414|
|  2|  414|
|  3|  414|
|  4|  415|
|  5|  414|
|  6|  414|
|  7|  414|
|  8|  414|
|  9|  414|
| 10|  414|
| 11|  414|
| 12|  414|
| 13|  414|
| 14|  414|
| 15|  414|
+---+-----+



## Writing them: one tiny file per partition

In [20]:
rare_orders.write.mode("overwrite").format("delta").saveAsTable("rare_orders_no_coalesce")

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 24, Finished, Available, Finished, False)

In [21]:
spark.sql("DESCRIBE DETAIL rare_orders_no_coalesce").select("numFiles", "sizeInBytes").show()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 25, Finished, Available, Finished, False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|      16|     135787|
+--------+-----------+



## Coalesce first: one clean file

In [22]:
from pyspark.sql import functions as F

orders = (spark.read.option("header", True).option("inferSchema", True)
              .csv("Files/orders.csv"))

orders = orders.repartition(16) 
# filter to something RARE -> now partitions really are near-empty
rare_orders = orders.filter(F.col("status") == "PENDING")  
print("partitions:", rare_orders.rdd.getNumPartitions())    # still 16

(rare_orders.withColumn("pid", F.spark_partition_id())
     .groupBy("pid").count().orderBy("pid").show())
# now you'll see counts like 400, 412, 414... -> THAT is "too tiny"

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 26, Finished, Available, Finished, False)

partitions: 16
+---+-----+
|pid|count|
+---+-----+
|  0|  414|
|  1|  414|
|  2|  414|
|  3|  414|
|  4|  415|
|  5|  414|
|  6|  414|
|  7|  414|
|  8|  414|
|  9|  414|
| 10|  414|
| 11|  414|
| 12|  414|
| 13|  414|
| 14|  414|
| 15|  414|
+---+-----+



In [23]:
rare_orders.coalesce(1).write.mode("overwrite").format("delta").saveAsTable("rare_orders_coalesced")

spark.sql("DESCRIBE DETAIL rare_orders_coalesced").select("numFiles", "sizeInBytes").show()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 27, Finished, Available, Finished, False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       1|      88979|
+--------+-----------+



<div style="font-size: 1.4em;">

<h2>When to use which</h2>

<ul>
<li><b><code>coalesce</code></b> (local merge, no shuffle, reduce-only) → after a heavy filter that leaves many near-empty partitions: merge down before writing so you don't produce hundreds of tiny files.</li>
<li><b><code>repartition</code></b> (full shuffle, up or down, evenly balanced) → when a read gives you too few partitions (poor parallelism), or you need even distribution before a join/groupBy.</li>
</ul>

</div>

# Two actions, NO cache (recompute twice)
### Two jobs, and both do the full filter → groupBy work from the source. The second action gained nothing from the first — Spark recomputed the entire lineage.

In [24]:
from pyspark.sql import functions as F

orders = spark.read.table("orders_big")

a = (orders
     .filter(F.col("status") == "CANCELLED")
     .groupBy("user_id").agg(F.sum("amount").alias("total")))

a.write.format("noop").mode("overwrite").save() # ACTION 1 -> runs the whole pipeline
a.count()     # ACTION 2 -> runs the WHOLE pipeline AGAIN (nothing reused)

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 28, Finished, Available, Finished, False)

500

# Same thing, WITH cache
### Still two jobs (caching doesn't reduce job count), but now the second job's map-side stages show as Skipped — it reads the cached result instead of recomputing. Same result, cheaper second action.

In [25]:
from pyspark.sql import functions as F

orders = spark.read.table("orders_big")

b = (orders
     .filter(F.col("status") == "CANCELLED")
     .groupBy("user_id").agg(F.sum("amount").alias("total")))

b.cache()     # mark for caching (lazy — nothing stored yet)

b.write.format("noop").mode("overwrite").save() # ACTION 1 -> runs the pipeline AND fills the cache
b.count()     # ACTION 2 -> reads from cache, skips the filter/groupBy work

# b.unpersist()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 29, Finished, Available, Finished, False)

500

#### **Cache when you'll run multiple actions on the same DataFrame** — the expensive lineage runs once and later actions reuse it. If you only use a DataFrame once, caching just wastes memory: the store costs more than it saves.

# Data Skew on groupBy/aggregations

## Build a skewed table (one hot key)

In [26]:
from pyspark.sql import functions as F

# 90% of rows share user_id = 1 (the hot key)
skewed_df = (spark.range(0, 2_000_000)
          .withColumn("user_id",
              F.when(F.col("id") % 10 != 0, F.lit(1))
               .otherwise((F.col("id") % 499) + 2))
          .withColumn("amount", (F.col("id") % 500).cast("double")))

display(skewed_df)

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 08ae187a-cddd-4db7-905d-425dea18e76b)

## DETECT it (the skill to actually learn):

In [27]:
shuffled_df = skewed_df.repartition(8, "user_id")   # shuffle by the skewed key

(shuffled_df.withColumn("pid", F.spark_partition_id())
         .groupBy("pid").count().orderBy(F.col("count").desc())
         .show())

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 31, Finished, Available, Finished, False)

+---+-------+
|pid|  count|
+---+-------+
|  5|1820038|
|  7|  28855|
|  1|  27657|
|  3|  26454|
|  2|  25255|
|  6|  25249|
|  0|  24047|
|  4|  22445|
+---+-------+



## Make the skew show as a slow task

In [28]:
# force real work across the shuffled partitions so the skew shows as a slow task
(shuffled_df.groupBy("user_id")
         .agg(F.sum("amount").alias("total"))
         .write.format("noop").mode("overwrite").save())

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 32, Finished, Available, Finished, False)

## How to recognize skew

One partition holds far more rows than the others (`spark_partition_id` count), or in the Spark UI one task takes much longer than the rest (a **straggler**) while other cores sit idle.

**Cause:** a *hot key* — one value appears far more often, so a shuffle piles all its rows into one partition.

**Fixing it manually** is possible — a technique called *salting* splits the hot key across partitions — but it's fiddly: you rewrite your aggregation in two steps by hand. Fortunately, **AQE handles skew automatically**, with no code change.

![salting](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/salting.png)

# The fix with Salting

In [ ]:
from pyspark.sql import functions as F

SALT = 8   # split the hot key into 8 sub-keys

# Step 1: add a random salt, then group by (user_id, salt)
#         user_id=1's rows now split across 8 sub-groups instead of piling into one
salted_df = skewed_df.withColumn("salt", (F.rand() * SALT).cast("int"))

partial_df = (salted_df.groupBy("user_id", "salt")
                 .agg(F.sum("amount").alias("partial_total")))

# Step 2: drop the salt and sum the partials back into the true per-user total
final_df = (partial_df.groupBy("user_id")
                .agg(F.sum("partial_total").alias("total")))

final_df.write.format("noop").mode("overwrite").save()

# Data Skew on joins

## AQE off


In [ ]:
# Turn AQE fully off so stage/task counts match the theory
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")   # make the default explicit

# verify
print("aqe:     ", spark.conf.get("spark.sql.adaptive.enabled"))
print("coalesce:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

## Build the two tables

In [29]:
from pyspark.sql import functions as F

# big table, hot key user_id = 1 (same skew as before)
skewed_df = (spark.range(0, 2_000_000)
          .withColumn("user_id",
              F.when(F.col("id") % 10 != 0, F.lit(1))
               .otherwise((F.col("id") % 499) + 2))
          .withColumn("amount", (F.col("id") % 500).cast("double")))

# small dimension table to join against, one row per user
dim_df = (spark.range(1, 501)
       .withColumnRenamed("id", "user_id")
       .withColumn("region", F.concat(F.lit("region_"), (F.col("user_id") % 5))))
       
display(dim_df)

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eed700e0-6ecf-4200-8675-ca0c7af7bdcb)

## The skewed join (forced sort-merge join)

In [30]:
joined = skewed_df.join(dim_df.hint("MERGE"), "user_id")   # MERGE = sort-merge join
joined.write.format("noop").mode("overwrite").save()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 34, Finished, Available, Finished, False)

![aqe](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/aqe.png)

## AQE on this time

In [31]:
from pyspark.sql import functions as F

# turn AQE on + skew-join handling (this is the whole point of the section)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "10MB") # just for demo purposes
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "2MB") # just for demo purposes

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 35, Finished, Available, Finished, False)

## Build the two tables

In [32]:
# big table, hot key user_id = 1 (same skew as before)
skewed_df = (spark.range(0, 2_000_000)
          .withColumn("user_id",
              F.when(F.col("id") % 10 != 0, F.lit(1))
               .otherwise((F.col("id") % 499) + 2))
          .withColumn("amount", (F.col("id") % 500).cast("double")))

# small dimension table to join against, one row per user
dim_df = (spark.range(1, 501)
       .withColumnRenamed("id", "user_id")
       .withColumn("region", F.concat(F.lit("region_"), (F.col("user_id") % 5))))

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 36, Finished, Available, Finished, False)

## The skewed join (forced sort-merge join)

In [33]:
joined = skewed_df.join(dim_df.hint("MERGE"), "user_id")   # MERGE = sort-merge join
joined.write.format("noop").mode("overwrite").save()

StatementMeta(, 71c4c63e-d03f-4351-913c-527cae8455a1, 37, Finished, Available, Finished, False)

## Handling data skew — pick the fix that matters for *your* operation

Skew shows up differently for **aggregations** (`groupBy`) vs **joins**, and the fix differs too:

| Operation | AQE helps? | What you actually do |
| --- | --- | --- |
| **`groupBy` / aggregation** | ❌ No — AQE won't split a hot aggregation key | **Salt** by hand (group by key+salt, then re-sum) |
| **Join** | ✅ Yes — AQE splits the skewed partition at runtime | Turn AQE on; or broadcast the small side to skip the shuffle |

## AQE Master Switches

In [ ]:
# --- AQE master switches ---
spark.conf.set("spark.sql.adaptive.enabled", "true")                          # turn AQE on (default: true in Spark 3.2+)

# --- 1. Coalesce post-shuffle partitions ---
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")       # merge small post-shuffle partitions
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")      # target size per partition after coalescing
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1m")# floor: don't make partitions smaller than this

# --- 2. Sort-merge -> broadcast at runtime ---
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", "10m")        # switch to broadcast if a side ends up under this

# --- 3. Skew join handling ---
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")                 # split skewed partitions in joins
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")      # "skewed" = > 5x the median partition
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256m")  # AND bigger than this absolute size

![r](https://raw.githubusercontent.com/ssteinau/public_speaking/main/spark_internals_1/images/getting-started-with-spark-internals_steinau-steinau_1241314_feedback-code.png)

In [ ]:
ssteinau public_speaking